
# Minimal RAG PoC — Clean Version (Local Embeddings + FAISS, OpenAI Generation)

This cleaned notebook removes redundancies and consolidates logic into a single clear flow:

- **Embeddings**: local (`sentence-transformers/all-MiniLM-L6-v2`) for both documents & queries
- **Index**: FAISS (inner product on L2-normalized vectors)
- **Retrieval**: top-k chunk search
- **Generation**: OpenAI chat model (`gpt-4o-mini`) with graceful fallback if rate-limited
- **Eval**: expected-substring check + citation sanity (citations must reference retrieved chunks)

> Set `OPENAI_API_KEY` if you want to use the OpenAI generation step. Otherwise, the fallback summary will kick in.


## 1) Setup

In [11]:

# If needed, install once:
# %pip install --upgrade sentence-transformers faiss-cpu numpy pandas tqdm openai


## 2) Imports & configuration

In [12]:

import os
import re
from typing import List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import faiss

# Local embeddings
from sentence_transformers import SentenceTransformer

# OpenAI for generation only (optional)
from openai import OpenAI, RateLimitError

# ----- Config -----
EMBED_MODEL_LOCAL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "gpt-4o-mini"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 60
TOP_K = 4
TEMPERATURE = 0.2

# Init clients/models (OpenAI is optional; only used if key is present)
openai_enabled = bool(os.getenv("OPENAI_API_KEY"))
if openai_enabled:
    client = OpenAI()

embedder = SentenceTransformer(EMBED_MODEL_LOCAL)  # used for docs + queries


## 3) Sample documents (~10)

In [13]:

docs = [
    ("Astronomy Notes", 
     "Stars form in molecular clouds. The lifecycle of a star depends on its mass. "
     "Massive stars end as supernovae, while Sun-like stars become white dwarfs."),

    ("Home Coffee Guide", 
     "Use freshly roasted beans. Grind size affects extraction: finer for espresso, coarser for French press. "
     "Water temperature around 92-96°C usually works well."),

    ("Indoor Plants 101", 
     "Snake plants tolerate low light and infrequent watering. Peace lilies like indirect light and moist soil. "
     "Rotate plants for even growth."),

    ("Project Management Tips", 
     "Define scope clearly and prioritize tasks. Short iterations with demos help reduce risk. "
     "Use retrospectives to improve team processes."),

    ("Python Tricks", 
     "List comprehensions are concise. Generators are memory-efficient. "
     "The standard library includes powerful modules like itertools and functools."),

    ("Healthy Sleep", 
     "Consistent sleep schedules align circadian rhythms. Reduce blue light before bed. "
     "A cool, dark, quiet room supports better sleep."),

    ("Running Basics", 
     "Increase weekly mileage gradually to avoid injury. Alternate hard and easy days. "
     "Proper shoes and strength training improve performance."),

    ("Budget Cooking", 
     "Plan meals, buy staples in bulk, and cook once for multiple meals. "
     "Use seasonal produce and freeze leftovers."),

    ("Basic First Aid", 
     "For small cuts, clean the wound and apply a sterile bandage. For burns, cool under running water. "
     "Know when to seek medical help."),

    ("Travel Packing", 
     "Use a packing list and roll clothes to save space. Separate liquids in a clear bag. "
     "Carry essentials like meds and chargers in your personal item.")
]

docs_df = pd.DataFrame(docs, columns=["title", "text"])
docs_df


,title,text
0,Astronomy Notes,Stars form in molecular clouds. The lifecycle ...
1,Home Coffee Guide,Use freshly roasted beans. Grind size affects ...
2,Indoor Plants 101,Snake plants tolerate low light and infrequent...
3,Project Management Tips,Define scope clearly and prioritize tasks. Sho...
4,Python Tricks,List comprehensions are concise. Generators ar...
5,Healthy Sleep,Consistent sleep schedules align circadian rhy...
6,Running Basics,Increase weekly mileage gradually to avoid inj...
7,Budget Cooking,"Plan meals, buy staples in bulk, and cook once..."
8,Basic First Aid,"For small cuts, clean the wound and apply a st..."
9,Travel Packing,Use a packing list and roll clothes to save sp...


## 4) Chunking

In [14]:

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    out = []
    start = 0
    n = len(text)
    while start < n:
        end = min(n, start + chunk_size)
        out.append(text[start:end])
        if end == n:
            break
        start = max(0, end - overlap)
    return out

corpus = []
for i, row in docs_df.iterrows():
    chs = chunk_text(row["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for j, ch in enumerate(chs):
        corpus.append({
            "doc_id": i,
            "doc_title": row["title"],
            "chunk_id": j,
            "text": ch
        })

corpus_df = pd.DataFrame(corpus)
print("Total chunks:", len(corpus_df))
corpus_df.head()


Total chunks: 10


,doc_id,doc_title,chunk_id,text
0,0,Astronomy Notes,0,Stars form in molecular clouds. The lifecycle ...
1,1,Home Coffee Guide,0,Use freshly roasted beans. Grind size affects ...
2,2,Indoor Plants 101,0,Snake plants tolerate low light and infrequent...
3,3,Project Management Tips,0,Define scope clearly and prioritize tasks. Sho...
4,4,Python Tricks,0,List comprehensions are concise. Generators ar...


## 5) Embeddings (local) + FAISS index

In [15]:

def embed_local(texts: List[str]) -> np.ndarray:
    # normalize_embeddings=True returns L2-normalized vectors
    arr = embedder.encode(texts, normalize_embeddings=True)
    return np.asarray(arr, dtype="float32")

chunk_texts = corpus_df["text"].tolist()
embeddings = embed_local(chunk_texts)

# FAISS index (inner product on normalized vectors ≈ cosine similarity)
faiss.normalize_L2(embeddings)  # safe even if already normalized
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print("FAISS index size:", index.ntotal)


FAISS index size: 10


## 6) Retrieval helpers

In [16]:

def retrieve(query: str, k: int = TOP_K) -> Tuple[np.ndarray, np.ndarray]:
    q = embed_local([query])[0]  # normalized
    q = q.reshape(1, -1)
    scores, idxs = index.search(q, k)
    return scores[0], idxs[0]

def build_context(idxs: List[int]) -> str:
    rows = corpus_df.iloc[idxs]
    return "\n\n".join(
        f"[{r['doc_title']}#chunk{int(r['chunk_id'])}] {r['text']}"
        for _, r in rows.iterrows()
    )


## 7) Prompting and RAG answer

In [17]:

SYSTEM_PROMPT = (
    "You are a concise, helpful assistant. "
    "Answer the user's question using ONLY the provided context. "
    "If the answer cannot be found in the context, say you do not know. "
    "Cite titles and chunk ids like [Title#chunkN]."
)

def build_user_prompt(query: str, context: str) -> str:
    return (
        "Context:\n" + context + "\n\n"
        "User question: " + query + "\n\n"
        "Instructions:\n"
        "- Use only the context.\n"
        "- If insufficient, say you do not know.\n"
        "- Include short inline citations like [Title#chunkN].\n"
    )

def answer_with_rag(query: str, k: int = TOP_K, temperature: float = TEMPERATURE):
    scores, idxs = retrieve(query, k=k)
    context = build_context(idxs)
    user_prompt = build_user_prompt(query, context)

    if openai_enabled:
        try:
            chat = client.chat.completions.create(
                model=GEN_MODEL,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            answer = chat.choices[0].message.content
        except RateLimitError:
            answer = (
                "OpenAI rate-limited; returning a context summary instead:\n\n"
                + context[:1200] + ("\n…[truncated]" if len(context) > 1200 else "")
            )
    else:
        answer = (
            "OPENAI_API_KEY not set; returning a context summary instead:\n\n"
            + context[:1200] + ("\n…[truncated]" if len(context) > 1200 else "")
        )

    return {
        "query": query,
        "answer": answer,
        "indices": idxs.tolist(),
        "scores": scores.tolist(),
        "context": context,
    }

# Example
example_query = "Does reducing light help with sleep?"
res = answer_with_rag(example_query)
print("Q:", res["query"])
print("\n--- Retrieved context ---\n", res["context"][:800])
print("\n--- Answer ---\n", res["answer"])


Q: Does reducing light help with sleep?

--- Retrieved context ---
 [Healthy Sleep#chunk0] Consistent sleep schedules align circadian rhythms. Reduce blue light before bed. A cool, dark, quiet room supports better sleep.

[Indoor Plants 101#chunk0] Snake plants tolerate low light and infrequent watering. Peace lilies like indirect light and moist soil. Rotate plants for even growth.

[Running Basics#chunk0] Increase weekly mileage gradually to avoid injury. Alternate hard and easy days. Proper shoes and strength training improve performance.

[Travel Packing#chunk0] Use a packing list and roll clothes to save space. Separate liquids in a clear bag. Carry essentials like meds and chargers in your personal item.

--- Answer ---
 Yes, reducing blue light before bed helps with sleep [Healthy Sleep#chunk0].


## 8) Simple evaluation

In [18]:

qa_gold = [
    {
        "q": "What water temperature is good for brewing coffee?",
        "expect_substring": "92-96",
        "expect_title": "Home Coffee Guide",
    },
    {
        "q": "Name a plant that can tolerate low light.",
        "expect_substring": "Snake",
        "expect_title": "Indoor Plants 101",
    },
    {
        "q": "How do very massive stars end their lives?",
        "expect_substring": "supernova",
        "expect_title": "Astronomy Notes",
    },
]

citation_pattern = re.compile(r"\[([^\[\]#]+)#chunk(\d+)\]")

def parse_citations(text: str):
    cites = []
    for m in citation_pattern.finditer(text or ""):
        title = m.group(1).strip()
        try:
            cid = int(m.group(2))
        except Exception:
            cid = None
        cites.append((title, cid))
    return cites

def eval_one(qitem):
    out = answer_with_rag(qitem["q"])
    # Expected substring check
    has_expected = qitem["expect_substring"].lower() in (out["answer"] or "").lower()

    # Citation sanity: cited chunks must be among retrieved
    retrieved_rows = corpus_df.iloc[out["indices"]]
    retrieved_keys = {(r["doc_title"], int(r["chunk_id"])) for _, r in retrieved_rows.iterrows()}
    cited = parse_citations(out["answer"])
    cited_keys = {(t, c) for (t, c) in cited if c is not None}

    citations_within_topk = len(cited_keys & retrieved_keys) == len(cited_keys) if cited_keys else True
    expected_title_cited = any(t == qitem["expect_title"] for (t, _) in cited)

    return {
        "question": qitem["q"],
        "expected_substring": qitem["expect_substring"],
        "has_expected_substring": has_expected,
        "citations_within_topk": citations_within_topk,
        "expected_title_cited": expected_title_cited,
        "citations": list(cited),
        "answer": out["answer"],
    }

rows = [eval_one(q) for q in qa_gold]
pd.DataFrame(rows)[["question","has_expected_substring","citations_within_topk","expected_title_cited","citations"]]


,question,has_expected_substring,citations_within_topk,expected_title_cited,citations
0,What water temperature is good for brewing cof...,True,True,True,"[(Home Coffee Guide, 0)]"
1,Name a plant that can tolerate low light.,True,True,True,"[(Indoor Plants 101, 0)]"
2,How do very massive stars end their lives?,True,True,True,"[(Astronomy Notes, 0)]"



## 9) Notes

- This version removes redundant imports, duplicate embedding calls, and OpenAI embedding usage (which was rate-limited).
- Retrieval embeddings for both docs and queries are **local** and consistent (same model).
- Generation uses OpenAI if `OPENAI_API_KEY` is set; otherwise a context summary is returned so the PoC still runs.
- Swap `GEN_MODEL` if you want a different chat model.
- To go fully offline, replace the generation step with a local LLM (e.g., via `ollama`).
